# Промышленность

## Описание проекта

Чтобы оптимизировать производственные расходы, металлургический комбинат «Стальная птица» решил уменьшить потребление электроэнергии на этапе обработки стали.  
Для этого комбинату нужно контролировать температуру сплава.  
Задача — построить модель, которая будет её предсказывать.  
Заказчик хочет использовать разработанную модель для имитации технологического процесса.  

## Описание процесса обработки

Сталь обрабатывают в металлическом ковше вместимостью около 100 тонн.  
Чтобы ковш выдерживал высокие температуры, изнутри его облицовывают огнеупорным кирпичом.  
Расплавленную сталь заливают в ковш и подогревают до нужной температуры графитовыми электродами. Они установлены на крышке ковша.  
Сначала происходит десульфурация — из стали выводят серу и корректируют её химический состав добавлением примесей.  
Затем сталь легируют — добавляют в неё куски сплава из бункера для сыпучих материалов или порошковую проволоку через специальный трайб-аппарат.  
Прежде чем в первый раз ввести легирующие добавки, специалисты производят химический анализ стали и измеряют её температуру.  
Потом температуру на несколько минут повышают, уже после этого добавляют легирующие материалы и продувают сталь инертным газом, чтобы перемешать, а затем снова проводят измерения.  
Такой цикл повторяется до тех пор, пока не будут достигнуты нужный химический состав стали и оптимальная температура плавки.  
Дальше расплавленная сталь отправляется на доводку металла или поступает в машину непрерывной разливки.  
Оттуда готовый продукт выходит в виде заготовок-слябов (англ. slab, «плита»).

## Описание данных

Данные хранятся в Sqlite  — СУБД, в которой база данных представлена одним файлом.  
Она состоит из нескольких таблиц:  
- steel.data_arc — данные об электродах;  
- steel.data_bulk — данные об объёме сыпучих материалов;  
- steel.data_bulk_time — данные о времени подачи сыпучих материалов;  
- steel.data_gas — данные о продувке сплава газом;  
- steel.data_temp — данные об измерениях температуры;  
- steel.data_wire — данные об объёме проволочных материалов;  
- steel.data_wire_time — данные о времени подачи проволочных материалов.  

Таблица steel.data_arc  
- key — номер партии;
- BeginHeat — время начала нагрева;
- EndHeat — время окончания нагрева;
- ActivePower — значение активной мощности;
- ReactivePower — значение реактивной мощности.

Таблица steel.data_bulk
- key — номер партии;
- Bulk1 … Bulk15 — объём подаваемого материала.  

Таблица steel.data_bulk_time  
- key — номер партии;
- Bulk1 … Bulk15 — время подачи материала.

Таблица steel.data_gas
- key — номер партии;
- gas — объём подаваемого газа.

Таблица steel.data_temp
- key — номер партии;
- MesaureTime — время замера;
- Temperature — значение температуры.

Таблица steel.data_wire
- key — номер партии;
- Wire1 … Wire9 — объём подаваемых проволочных материалов.

Таблица steel.data_wire_time
- key — номер партии;
- Wire1 … Wire9 — время подачи проволочных материалов.  

Во всех файлах столбец key содержит номер партии.  
В таблицах может быть несколько строк с одинаковым значением key: они соответствуют разным итерациям обработки.

# Проектирование

In [ ]:
# !pip install catboost -q

## Импорты

In [ ]:
# ============================================
# СТАНДАРТНЫЕ БИБЛИОТЕКИ PYTHON
# ============================================
import os
import re
import logging
import warnings
import random
from tqdm import tqdm
from IPython.display import HTML, display


# ============================================
# ОСНОВНЫЕ БИБЛИОТЕКИ ДЛЯ АНАЛИЗА ДАННЫХ
# ============================================
import numpy as np
import pandas as pd

# ============================================
# ВИЗУАЛИЗАЦИЯ
# ============================================
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

# ============================================
# РАБОТА С БАЗАМИ ДАННЫХ
# ============================================
from sqlalchemy import create_engine, inspect, Table, MetaData
from sqlalchemy.orm import sessionmaker

# ============================================
# ПРЕДОБРАБОТКА ДАННЫХ И СТАТИСТИКА
# ============================================
from sklearn.preprocessing import RobustScaler, StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ============================================
# КОРРЕЛЯЦИОННЫЙ АНАЛИЗ (PHIK)
# ============================================
import phik
from phik import phik_matrix

# ============================================
# РАЗДЕЛЕНИЕ ДАННЫХ, КРОСС-ВАЛИДАЦИЯ, ОТБОР ПРИЗНАКОВ
# ============================================
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.feature_selection import SelectKBest, f_regression

# ============================================
# МЕТРИКИ КАЧЕСТВА
# ============================================
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# ============================================
# МОДЕЛИ МАШИННОГО ОБУЧЕНИЯ
# ============================================
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
import lightgbm as lgb

# ============================================
# ПАЙПЛАЙНЫ
# ============================================
from sklearn.pipeline import Pipeline

# ============================================
# ОПТИМИЗАЦИЯ ГИПЕРПАРАМЕТРОВ (OPTUNA)
# ============================================
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

# ============================================
# ГЛУБОКОЕ ОБУЧЕНИЕ (PYTORCH)
# ============================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ============================================
# НАСТРОЙКИ ЛОГИРОВАНИЯ И ПРЕДУПРЕЖДЕНИЙ
# ============================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

# Отключаем лишние предупреждения
warnings.filterwarnings("ignore")

## Константы

In [ ]:
BD = '../data/ds-plus-final.db'
RANDOM_STATE = 10326
THRESHOLD = 0.3 # это для удаления пропусков, где больше N%
TEST_SIZE = 0.25
TARGET = 'конечная_температура'
N_TRIALS_NN = 5
CV_FOLDS = 5
EPOCHS = 10000
EARLY_STOP = 10

N_TRIALS_CB = 100
CAT_EARLY_STOP = 20
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(DEVICE)

## Функции проекта

In [ ]:
# сделаем функцию оценки пропусков в датасетах
def missing_data(data):
    missing_data = data.isna().sum()
    missing_data = missing_data[missing_data > 0]
    display(missing_data)


# функция для обработки пробелов
def process_spaces(s):
    if isinstance(s, str):
        s = s.strip()
        s = " ".join(s.split())
    return s


# замена пробелов на нижнее подчеркинвание в названии столбцов
def replace_spaces(s):
    if isinstance(s, str):
        s = s.strip()
        s = "_".join(s.split())
    return s


def drop_duplicated(data):
    # проверка дубликатов
    logger.info("Проверим дубликаты и удалим, если есть")
    num_duplicates = data.duplicated().sum()
    display(num_duplicates)

    if num_duplicates > 0:
        logger.info("Удаляем")
        data = data.drop_duplicates(keep="first").reset_index(
            drop=True
        )  # обновляем DataFrame
    else:
        logger.info("Дубликаты отсутствуют")
    return data


def normalize_columns(columns):
    new_cols = []
    for col in columns:
        # вставляем "_" перед заглавной буквой (латиница или кириллица), кроме первой
        col = re.sub(r"(?<!^)(?=[A-ZА-ЯЁ])", "_", col)
        # приводим к нижнему регистру
        col = col.lower()
        new_cols.append(col)
    return new_cols


def check_data(data):
    # приведем все к нижнему регистру
    data.columns = normalize_columns(data.columns)

    # удалим лишние пробелы в строках
    for col in data.columns:
        if data[col].dtype == "object":
            data[col] = data[col].apply(
                lambda x: process_spaces(x) if isinstance(x, str) else x
            )

    # и в названии столбцов
    data.columns = [replace_spaces(col) for col in data.columns]

    # строки в ячейках строчными буквами
    for col in data.columns:
        if data[col].dtype == "object":
            # Безопасное преобразование: только для строк, игнорируем None и не-строки
            data[col] = data[col].apply(
                lambda x: x.lower() if isinstance(x, str) else x
            )

    # общая информация
    logger.info("Общая информация базы данных")
    display(data.info())

    # 5 строк
    logger.info("5 случайных строк")
    display(data.sample(5))

    # пропуски
    logger.info("Число пропусков в базе данных")
    display(missing_data(data))

    # проверка на наличие пропусков
    if data.isnull().sum().sum() > 0:
        logger.info("Визуализация пропусков")
        msno.bar(data)
        plt.show()

    # средние характеристики
    logger.info("Характеристики базы данных")
    display(data.describe().T)

    # data = drop_duplicated(data)

    return data  # возвращаем измененные данные

def plot_combined(data, col=None, target=None, col_type=None, legend_loc='best'):
    """
    Строит графики для числовых столбцов в DataFrame, автоматически определяя их типы (дискретные или непрерывные).

    :param data: DataFrame, содержащий данные для визуализации.
    :param col: Список столбцов для построения графиков. Если None, будут использованы все числовые столбцы.
    :param target: Столбец, по которому будет производиться разделение (для hue в графиках).
    :param col_type: Словарь, определяющий типы столбцов ('col' для непрерывных и 'dis' для дискретных).
                     Если None, типы будут определены автоматически.
    :param legend_loc: Положение легенды для графиков (по умолчанию 'best').
    :return: None. Графики отображаются с помощью plt.show().
    """
    
    # Определяем числовые столбцы
    if col is None:
        numerical_columns = data.select_dtypes(include=['int', 'float']).columns.tolist()
    else:
        numerical_columns = col

    # Если col_type не указан, определяем типы автоматически
    if col_type is None:
        col_type = {}
        for col in numerical_columns:
            unique_count = data[col].nunique()
            if unique_count > 20:
                col_type[col] = 'col'  # Непрерывные данные
            else:
                col_type[col] = 'dis'  # Дискретные данные

    total_plots = len(numerical_columns) * 2
    ncols = 2
    nrows = (total_plots + ncols - 1) // ncols

    fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(12, 5 * nrows))
    axs = axs.flatten()

    index = 0

    for col in numerical_columns:
        # Определяем тип графика
        plot_type = col_type.get(col)
        if plot_type is None:
            raise ValueError(f"Тип для столбца '{col}' не указан в col_type.")

        # Гистограмма или countplot
        if index < len(axs):
            if plot_type == 'col':
                if target is not None:
                    sns.histplot(data, x=col, hue=target, bins=20, kde=True, ax=axs[index])
                    handles, labels = axs[index].get_legend_handles_labels()
                    if handles:
                        axs[index].legend(title=target, loc=legend_loc)
                else:
                    sns.histplot(data[col].dropna(), bins=20, kde=True, ax=axs[index])
                axs[index].set_title(f'Гистограмма: {col}')
            elif plot_type == 'dis':
                if target is not None:
                    sns.countplot(data=data, x=col, hue=target, ax=axs[index])
                    handles, labels = axs[index].get_legend_handles_labels()
                    if handles:
                        axs[index].legend(title=target, loc=legend_loc)
                else:
                    sns.countplot(data=data, x=col, ax=axs[index])
                axs[index].set_title(f'Countplot: {col}')
            index += 1

        # Боксплот
        if index < len(axs):
            sns.boxplot(x=data[col], ax=axs[index])
            axs[index].set_title(f'Боксплот: {col}')
            index += 1

    # Отключаем оставшиеся оси
    for j in range(index, len(axs)):
        axs[j].axis('off')

    plt.tight_layout()
    plt.show()
    
def clear_df_missing_cols(df):
    missing_ratio = df.isna().mean()
    cols_to_keep = missing_ratio[missing_ratio <= THRESHOLD].index.tolist()
    df = df[cols_to_keep]

    logger.info(f"Удалено признаков: {len(missing_ratio) - len(cols_to_keep)}")
    logger.info(f"Осталось признаков: {len(cols_to_keep)}")    
    return df

## Загрузка данных

### Подключение движка

In [ ]:
metadata = MetaData()
engine = create_engine(f'sqlite:///{BD}', echo=False)
inspector = inspect(engine)
Session = sessionmaker(engine)

### Исследовательский анализ

#### Данные в БД

In [ ]:
# посмотрим что есть в БД
table_names = inspector.get_table_names()
logger.info(table_names)

В нашем файлике есть еще и "левые данные" по второму проекту  
Ну да пусть будут, нам нужны только с префикмом data_ 


#### Наличие данных в таблицах

In [ ]:
# объявим переменные
data_arc = Table('data_arc', metadata, autoload_with=engine)
data_bulk = Table('data_bulk', metadata, autoload_with=engine)
data_bulk_time = Table('data_bulk_time', metadata, autoload_with=engine)
data_gas = Table('data_gas', metadata, autoload_with=engine)
data_temp = Table('data_temp', metadata, autoload_with=engine)
data_wire = Table('data_wire', metadata, autoload_with=engine)
data_wire_time = Table('data_wire_time', metadata, autoload_with=engine)

In [ ]:
# и преобразуем в пандас для удобства работы
df_arc = pd.read_sql(data_arc.select(), engine)
df_bulk = pd.read_sql(data_bulk.select(), engine)
df_bulk_time = pd.read_sql(data_bulk_time.select(), engine)
df_gas = pd.read_sql(data_gas.select(), engine)
df_temp = pd.read_sql(data_temp.select(), engine)
df_wire = pd.read_sql(data_wire.select(), engine)
df_wire_time = pd.read_sql(data_wire_time.select(), engine)

##### df_arc

1. Изучить пропуски и аномалии, распределение признаков;  
2. Скорректировать или удалить партии с аномальными значениями;  
3. Генерация новых признаков - длительность нагрева, общую мощность, соотношение активной мощности к реактивной, количество запуска нагрева электродами;  

In [ ]:
df_arc = check_data(df_arc)

Пропусков нет  
Даты надо преобразовать в нормальный тип  

Что проверить  
1) сопоставление дат - начало должно быть раньше конца  

In [ ]:
df_arc['начало_нагрева_дугой'] = df_arc['начало_нагрева_дугой'].astype('datetime64[ns]')
df_arc['конец_нагрева_дугой'] = df_arc['конец_нагрева_дугой'].astype('datetime64[ns]')

In [ ]:
df_arc['продолжительность_нагрева_сек'] = (df_arc['конец_нагрева_дугой'] - df_arc['начало_нагрева_дугой']).dt.total_seconds()

In [ ]:
df_arc.describe().T

In [ ]:
# теперь посмотрим распределение данных
plot_combined(df_arc, col=None, target=None, col_type=None, legend_loc='best')

Везде наблюдаем наличие сильных выбросов, проверим детальнее  
К тому же есть отрицательные значения чего не должно быть

In [ ]:
df_arc[df_arc['активная_мощность'] > 1.3]

Значения увеличиваются равномерно и постепенно, сложно сказать ошибка это или нет

In [ ]:
df_arc[df_arc['реактивная_мощность'] < 0]

Явная ошибка заполнения данных, посчитаем среднюю активная_мощность и среднюю реактивная_мощность  
Посмотрим коэффицент изменения и изменим эту ошибку соответственно

In [ ]:
avg_act_power = df_arc['активная_мощность'].mean()
avg_react_power = df_arc['реактивная_мощность'].mean()
koef_power = (avg_act_power / avg_react_power) - 1
logger.info(avg_act_power)
logger.info(avg_react_power)
logger.info(koef_power)


In [ ]:
df_arc.loc[df_arc['реактивная_мощность'] < 0, 'реактивная_мощность'] = df_arc['активная_мощность'] * koef_power

In [ ]:
df_arc[df_arc['продолжительность_нагрева_сек'] > 400]

Здесь как будто бы тоже всё ок по времени, надо детальнее изучать вопрос относительно нормальности плавки

In [ ]:
# добавим немного новых фич
df_arc['полная_мощность'] = np.sqrt(df_arc['активная_мощность']**2 + df_arc['реактивная_мощность']**2)
df_arc['соотношение_мощностей'] = df_arc['реактивная_мощность'] / df_arc['активная_мощность']
df_arc['cos_phi'] = 1 / np.sqrt(1 + df_arc['соотношение_мощностей']**2)

In [ ]:
# теперь посмотрим распределение данных
plot_combined(df_arc, col=None, target=None, col_type=None, legend_loc='best')

In [ ]:
display(df_arc.info())
display(df_arc.head())
display(df_arc.describe().T)

Самое главное, надо переделать ключи в одну строку - в широкий формат  
Но только после того как создадим новые признаки

In [ ]:
df_arc['число_нагревов'] = df_arc.groupby('key').cumcount() + 1

df_arc['длительность'] = (
    pd.to_datetime(df_arc['конец_нагрева_дугой']) - 
    pd.to_datetime(df_arc['начало_нагрева_дугой'])
).dt.total_seconds()

avg_stats = df_arc.groupby('key').agg(
    средняя_активная_мощность=('активная_мощность', 'mean'),
    средняя_реактивная_мощность=('реактивная_мощность', 'mean'),
    среднее_время_нагрева_сек=('длительность', 'mean')
).reset_index()

df_pivot = df_arc.pivot(
    index='key',
    columns='число_нагревов',
    values=['начало_нагрева_дугой', 'конец_нагрева_дугой', 
            'активная_мощность', 'реактивная_мощность', 'длительность']
)

df_pivot.columns = [f'{col[0]}_{col[1]}' for col in df_pivot.columns]
df_pivot = df_pivot.reset_index()

df_pivot = df_pivot.merge(avg_stats, on='key', how='left')
df_arc = df_pivot

In [ ]:
logger.info("Одна плавка - одна строчка:")
display(df_arc.head())

In [ ]:
# и почистим от пропусков
df_arc = clear_df_missing_cols(df_arc)

In [ ]:
# и сразу заменим на 0 пропуски в активной и реактивной мощностях

power_cols = [col for col in df_arc.columns 
              if 'активная_мощность' in col or 'реактивная_мощность' in col]

missing_before = df_arc[power_cols].isna().sum().sum()
# df_arc[power_cols] = df_arc[power_cols].fillna(0)

##### df_bulk

In [ ]:
df_bulk = check_data(df_bulk)

Много пропусков - но это мелочи.  
Заменим на 0, т.к. это просто означает, что не было подачи сыпучего материала.  
Ну и тип данных везде поменять на float  

PS: после первых экспериментов попробуем убрать те загрузки, где больше 50% - пропуски

In [ ]:
df_bulk = clear_df_missing_cols(df_bulk)

In [ ]:
for col in df_bulk.columns:
    if col != 'key':
        df_bulk[col] = df_bulk[col].astype(float)

In [ ]:
df_bulk.describe().T

In [ ]:
# теперь посмотрим распределение данных
plot_combined(df_bulk, col=None, target=None, col_type=None, legend_loc='best')

Все добавки выше нуля - это главное  
а по числу добавок - сложно понять, что мы добавляем и сколько этого надо...

In [ ]:
display(df_bulk.info())
display(df_bulk.head())
display(df_bulk.describe().T)

In [ ]:
bulk_cols = [col for col in df_bulk.columns if col.startswith('bulk_')]
bulk_cols = sorted(bulk_cols, key=lambda x: int(x.split('_')[1]))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Тепловая карта засыпок
usage_matrix = pd.DataFrame({col: (df_bulk[col] > 0).astype(int) for col in bulk_cols})
sns.heatmap(usage_matrix.T, cmap='RdYlGn', cbar_kws={'label': 'Используется'}, ax=axes[0])
axes[0].set_title('Частота засыпок')
axes[0].set_xlabel('Номер плавки')
axes[0].set_ylabel('Номер засыпки')

# Распределение объемов
data_for_box = df_bulk[bulk_cols].melt(var_name='Номер засыпки', value_name='Объем')
data_for_box = data_for_box[data_for_box['Объем'] > 0]
sns.boxplot(data=data_for_box, x='Номер засыпки', y='Объем', ax=axes[1])
axes[1].set_title('Распределение объемов засыпки')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

Посмотрим отдельно на выброс засыпки в номере 12  
Именно номер плавки по порядку, непонятно

In [ ]:
df_bulk.head()

In [ ]:
display(df_bulk[df_bulk['bulk_12'] > 600])

Непонятно все равно, может быть и есть в этом какая-то логика, но мне она отсюда не видна.  
Лучше сделаем новые фичи:  
1) Суммарный объем засыпок;  
2) Число засыпок;  

In [ ]:
bulk_cols = [col for col in df_bulk.columns if 'bulk_' in col.lower()]
df_bulk['total_bulk'] = df_bulk[bulk_cols].sum(axis=1)

In [ ]:
df_bulk['count_bulk'] = df_bulk[bulk_cols].count(axis=1)

In [ ]:
df_bulk.head()

In [ ]:
# и посмотрим что вышло
cols = ['total_bulk', 'count_bulk']
plot_combined(df_bulk, col=cols, target=None, col_type=None, legend_loc='best')

In [ ]:
# df_bulk = df_bulk.fillna(0)

##### df_bulk_time

In [ ]:
df_bulk_time = check_data(df_bulk_time)

Аналогично и тут пропуски - время подачи, т.к. не было подачи по факту то и не было фиксации времени  
Здесь видимо пропуски будут NaT

In [ ]:
df_bulk_time.info()

In [ ]:
df_bulk_time.head()

Интересное наблюдение, отсчет идет с последней засыпке к 1й...   
Ну да пусть, нам бы посчитать суммарное время  

In [ ]:
df_bulk_time = clear_df_missing_cols(df_bulk_time)

In [ ]:
df_bulk_time

In [ ]:
bulk_time_cols = [col for col in df_bulk_time.columns if 'bulk_' in col.lower()]
df_bulk_time[bulk_time_cols] = df_bulk_time[bulk_time_cols].astype('datetime64[ns]')

In [ ]:
df_bulk_time['начало_засыпок'] = df_bulk_time[bulk_time_cols].min(axis=1)
df_bulk_time['конец_засыпок'] = df_bulk_time[bulk_time_cols].max(axis=1)
df_bulk_time['общее_время_засыпок'] = (df_bulk_time['конец_засыпок'] - df_bulk_time['начало_засыпок']).dt.total_seconds()

In [ ]:
# ну и средний интервал между подачами заодно

df_bulk_time['интервал_подачи_avg'] = (
    df_bulk_time[bulk_time_cols]
    .apply(lambda row: row.dropna().sort_values().diff().dt.total_seconds().mean(), axis=1)
)

In [ ]:
display(df_bulk_time.head())
display(df_bulk_time.info())
display(df_bulk_time.describe().T)

Вылезла неадекватная продолжительность подачи, посмотрим поближе

In [ ]:
# и посмотрим что вышло
cols = ['общее_время_засыпок', 'интервал_подачи_avg']
plot_combined(df_bulk_time, col=cols, target=None, col_type=None, legend_loc='best')

##### df_gas

In [ ]:
df_gas = check_data(df_gas)

Тут на первый взгляд все у нас хорошо  
но посмотрим и на график

In [ ]:
plot_combined(df_gas, col=None, target=None, col_type=None, legend_loc='best')

Не такой больше различие у максимума, но все равно посмотрим на бОльшие значения

In [ ]:
display(df_gas[df_gas['газ_1'] > 50])

Ну дали чутка больше газа и ладно, бывает

##### df_temp

In [ ]:
df_temp = check_data(df_temp)

У времени сделать нормальный тип данных  
Пропуски мы не сможем адекватно скорректировать  
Как эксперимент можем взять два соседних тайминга и просто взять среднюю между этими числами, критической проблемы тут быть не должно  
PS: изучив данные мы видим, что там на каждый key только один замер температур

In [ ]:
# df_temp['температура'] = df_temp['температура'].fillna(-1).astype('float64')
df_temp['температура'] = df_temp['температура'].astype('float64')

In [ ]:
df_temp['время_замера'] = df_temp['время_замера'].astype('datetime64[ns]')

In [ ]:
df_temp.info()

In [ ]:
temp_df_temp = df_temp[df_temp['температура'] > 0]
plot_combined(temp_df_temp, col=None, target=None, col_type=None, legend_loc='best')

In [ ]:
temp_df_temp.describe().T

В целом распределение температур выглядит адекватно

Что касается задания -  
1) таргет - последняя температура партии  
2) можно брать только те позиции, где есть начальная и конечная температура, т.е. минимум 2 позиции  
3) промежуточные - не брать, это утечка данных  
4) температура ниже 1500 - аномалия

In [ ]:
# посмотрим исходя из вводных от заказчика
# уберем сразу аномалии
df_temp = df_temp[df_temp['температура'] >= 1500]

Находим ключи, у которых 2 и более записей, т.к. это требования по ТЗ

In [ ]:
valid_keys = df_temp.groupby('key').size()
valid_keys = valid_keys[valid_keys >= 2].index

df_temp_filtered = df_temp[df_temp['key'].isin(valid_keys)]

df_temp_agg = df_temp_filtered.groupby('key').agg(
    время_первого_замера=('время_замера', 'min'),
    время_последнего_замера=('время_замера', 'max'),
    стартовая_температура=('температура', 'min'),
    конечная_температура=('температура', 'max')
).reset_index()

display(df_temp_agg.head())

In [ ]:
df_temp = df_temp_agg
display(df_temp)

##### df_wire

In [ ]:
df_wire = check_data(df_wire)

Тут как и с сыпучими материалами - либо добавили, либо нет.  
Вместо пропусков - 0

In [ ]:
for col in df_wire.columns:
    if col != 'key':
        df_wire[col] = df_wire[col].astype(float)

In [ ]:
# сразу добавим общуюю массу добавок
wire_cols = [col for col in df_wire.columns if 'wire_' in col.lower()]
df_wire['total_wire'] = df_wire[wire_cols].sum(axis=1)

In [ ]:
df_wire['count_wire'] = df_wire[wire_cols].count(axis=1)

In [ ]:
df_wire.head()

In [ ]:
df_wire = clear_df_missing_cols(df_wire)

In [ ]:
plot_combined(df_wire, col=None, target=None, col_type=None, legend_loc='best')

In [ ]:
display(df_wire.info())
display(df_wire.head())
display(df_wire.describe().T)

In [ ]:
wire_cols = [col for col in df_wire.columns if col.startswith('wire_')]
wire_cols = sorted(wire_cols, key=lambda x: int(x.split('_')[1]))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Тепловая карта засыпок
usage_matrix = pd.DataFrame({col: (df_wire[col] > 0).astype(int) for col in wire_cols})
sns.heatmap(usage_matrix.T, cmap='RdYlGn', cbar_kws={'label': 'Используется'}, ax=axes[0])
axes[0].set_title('Частота засыпок (wire)')
axes[0].set_xlabel('Номер плавки')
axes[0].set_ylabel('Номер засыпки')

# Распределение объемов
data_for_box = df_wire[wire_cols].melt(var_name='Номер засыпки', value_name='Объем')
data_for_box = data_for_box[data_for_box['Объем'] > 0]
sns.boxplot(data=data_for_box, x='Номер засыпки', y='Объем', ax=axes[1])
# axes[1].set_yscale('log')
axes[1].set_title('Распределение объемов засыпки (wire)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

Выборсы конечно же есть, но берем в расчте, что так надо  
В целом видно, что основная масса грузится в первые 2-3 этапа, а дальше уже мелкие корректировки

In [ ]:
# Добьем нулями
# df_wire = df_wire.fillna(0)

##### df_wire_time

In [ ]:
df_wire_time = check_data(df_wire_time)

In [ ]:
df_wire_time = clear_df_missing_cols(df_wire_time)

In [ ]:
wire_time_cols = [col for col in df_wire_time.columns if 'wire_' in col.lower()]
df_wire_time[wire_time_cols] = df_wire_time[wire_time_cols].astype('datetime64[ns]')

In [ ]:
df_wire_time.info()

In [ ]:
# если засыпок >2 то сделаем новые признаки
wire_time_cols = [col for col in df_wire_time.columns if col.startswith('wire_')]
wire_time_cols = sorted(wire_time_cols, key=lambda x: int(x.split('_')[1]))

In [ ]:
if len(wire_time_cols) >= 2:
    df_wire_time['начало_засыпок'] = df_wire_time[wire_time_cols].min(axis=1)
    df_wire_time['конец_засыпок'] = df_wire_time[wire_time_cols].max(axis=1)
    df_wire_time['общее_время_засыпок'] = (df_wire_time['конец_засыпок'] - df_wire_time['начало_засыпок']).dt.total_seconds()
    
    # Средний интервал между подачами (если подач меньше двух, будет NaN)
    df_wire_time['интервал_подачи_avg'] = (
        df_wire_time[wire_time_cols]
        .apply(lambda row: row.dropna().sort_values().diff().dt.total_seconds().mean(), axis=1)
    )
else:
    logger.info("Недостаточно колонок wire_ для расчета признаков (нужно >=2). Пропускаем.")

In [ ]:
display(df_wire_time.head())
display(df_wire_time.info())
display(df_wire_time.describe().T)

In [ ]:
# и посмотрим что вышло
if len(wire_time_cols) >= 2:
    cols = ['общее_время_засыпок', 'интервал_подачи_avg']
    plot_combined(df_wire_time, col=cols, target=None, col_type=None, legend_loc='best')

Где-то большие промежутки и общее время плавки добавок, но штош поделать...

##### Выводы

Все данные были изучены и проанализированы  
Обработаны пропуски  
Скорректированы типы данных  
Реализованы новые признаки  

#### Объединение данных

##### Объединение

In [ ]:
df_merged = (
    df_temp
    .merge(df_arc, on='key', how='inner')
    .merge(df_bulk, on='key', how='inner')
    .merge(df_bulk_time, on='key', how='inner', suffixes=('', '_bulk_time'))
    .merge(df_gas, on='key', how='inner')
    .merge(df_wire, on='key', how='inner')
    .merge(df_wire_time, on='key', how='inner', suffixes=('', '_wire_time'))
)

In [ ]:
display(df_merged.drop_duplicates())

In [ ]:
display(df_merged[df_merged['total_bulk'] == 0])
display(df_merged[df_merged['газ_1'] == 0])

Надо бы удалить временнЫе столбцы... они нам не понадобятся

In [ ]:
df_merged.info()

In [ ]:
non_datetime_cols = df_merged.select_dtypes(exclude=['datetime64[ns]']).columns
main_df = df_merged[non_datetime_cols].copy()

In [ ]:
main_df.head()

In [ ]:
main_df = main_df.fillna(0)

In [ ]:
main_df.info()

##### Исследуем итоговую таблицу

In [ ]:
df_cor = main_df.drop(columns=['key'])
correlation_matrix = df_cor.phik_matrix()

plt.figure(figsize=(20, 16))
sns.heatmap(correlation_matrix, 
            annot=True, 
            fmt=".2f", 
            cmap='coolwarm', 
            center=0,
            square=True, 
            linewidths=0.5,
            cbar_kws={'label': 'Phik correlation (|r| > 0.7)'},
            annot_kws={'size': 6})
plt.title('Сильные корреляции (|r| > 0.7)', fontsize=16, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

# ЗЫ: знаю, что лучше писать руками признаки, но мне так нравится больше + я проверяю, чтобы все было нормально

In [ ]:
threshold_cor = 0.9

features_to_drop = set()
corr_matrix = correlation_matrix.copy()
np.fill_diagonal(corr_matrix.values, 0)

for col in corr_matrix.columns:
    high_corr = corr_matrix[abs(corr_matrix[col]) > threshold_cor].index.tolist()
    if high_corr:
        features_to_drop.update(high_corr[1:])

logger.info(f"Удаляем признаков: {len(features_to_drop)}")
display("Список удаляемых признаков:", sorted(features_to_drop))

main_df = main_df.drop(columns=list(features_to_drop), errors='ignore')

logger.info(f"Осталось признаков: {main_df.shape[1] - 1}")

In [ ]:
# выведем корреляцию с таргетом
corr = correlation_matrix[TARGET].sort_values(ascending=False)

logger.info(f"Сильня корреляция с {TARGET}:")
display(corr[corr >= 0.9])

logger.info(f"Слабая корреляция с {TARGET}:")
display(corr[corr < 0.9])

In [ ]:
# # проверим зависимость VIF
# VIF = main_df.drop(columns=['key', TARGET])

# VIF = pd.get_dummies(VIF, drop_first=True)
# VIF = VIF.assign(const=1)

# for col in VIF.select_dtypes(include='bool').columns:
#     VIF[col] = VIF[col].astype(int)

# VIF = VIF.dropna()

# # Пересчитываем VIF
# vif_data = pd.DataFrame()
# vif_data["Признаки"] = VIF.columns
# vif_data["VIF"] = [variance_inflation_factor(VIF.values, i) for i in range(VIF.shape[1])]

# # Удаляем константу из результатов
# vif_data = vif_data[vif_data["Признаки"] != "const"]
# display(vif_data)

# Подготовка данных для VIF
vif_df = main_df.drop(columns=['key', TARGET])
vif_df = pd.get_dummies(vif_df, drop_first=True)
for col in vif_df.select_dtypes(include='bool').columns:
    vif_df[col] = vif_df[col].astype(int)
vif_df = vif_df.dropna()

# Список всех признаков
all_features = vif_df.columns.tolist()
features = all_features.copy()
threshold = 10

# Итеративное удаление признаков с VIF > threshold
while True:
    X_with_const = vif_df[features].assign(const=1)
    vif_vals = [variance_inflation_factor(X_with_const.values, i) for i in range(len(features))]
    max_vif = max(vif_vals)
    if max_vif <= threshold:
        break
    idx = vif_vals.index(max_vif)
    removed = features.pop(idx)
    logger.info(f"Удалён {removed} (VIF={max_vif:.2f})")

logger.info(f"Осталось признаков: {len(features)}")
logger.info(f"Список: {features}")

# Удаляем из main_df признаки, которые не прошли отбор
cols_to_drop = [col for col in all_features if col not in features]
if cols_to_drop:
    logger.info(f"Удаляем из main_df: {cols_to_drop}")
    main_df = main_df.drop(columns=cols_to_drop, errors='ignore')
else:
    logger.info("Ничего не удаляем, все признаки остались.")

По VIF всё больно - и как хорошо, что у нас нет линеек, а для всего остальное VIF чистить нет смысла, но мы все равно почистим  
Тогда просто делим данные и переходим к обучению

## Подготовим данные для обучения

In [ ]:
main_df.info()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    main_df.drop([TARGET, 'key'], axis=1),
    main_df[TARGET],
    test_size = TEST_SIZE, 
    random_state = RANDOM_STATE)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

In [ ]:
X_train_scaled.shape, X_test_scaled.shape

In [ ]:
X_train_scaled.describe().T

## Обучение

### случайный лес

In [ ]:
results = []

rf_params = {
    'n_estimators': 1000,
    'max_depth': 10,
    'min_samples_split': 5,
    'min_samples_leaf': 2,
    'max_features': 'sqrt',
    'bootstrap': True,
    'criterion': 'absolute_error',
    'random_state': RANDOM_STATE,
    'n_jobs': -1
}
model_rf = RandomForestRegressor(**rf_params)

# Кросс-валидация
cv_mae_scores = -cross_val_score(model_rf, X_train_scaled, y_train, cv=CV_FOLDS, 
                                  scoring='neg_mean_absolute_error', n_jobs=-1)
cv_r2_scores = cross_val_score(model_rf, X_train_scaled, y_train, cv=CV_FOLDS, 
                                scoring='r2', n_jobs=-1)

# Обучение на всей выборке
model_rf.fit(X_train_scaled, y_train)
y_train_pred_rf = model_rf.predict(X_train_scaled)

results.append({
    'model_name': 'RandomForestRegressor',
    'train_mae': mean_absolute_error(y_train, y_train_pred_rf),
    'train_r2': r2_score(y_train, y_train_pred_rf),
    'cv_mae_mean': cv_mae_scores.mean(),
    'cv_mae_std': cv_mae_scores.std(),
    'cv_r2_mean': cv_r2_scores.mean(),
    'cv_r2_std': cv_r2_scores.std(),
})

logger.info("Результаты:")
last_model = results[-1]
logger.info(f"Модель: {last_model['model_name']}")
for key, value in last_model.items():
    if key != 'model_name':
        logger.info(f"------- {key}: {value:.4f}" if isinstance(value, float) else f"------- {key}: {value}")

Немного не дотянули до требуемых значений

### catboost

In [ ]:
cb_params = {
    'iterations': 700,
    'depth': 4,
    'learning_rate': 0.05,
    'l2_leaf_reg': 5,
    'bagging_temperature': 0.8,
    'random_strength': 0.5,
    'loss_function': 'MAE',
    'random_seed': RANDOM_STATE,
    'allow_writing_files': False,
    'verbose': False,
    'thread_count': -1
}
model_cb = CatBoostRegressor(**cb_params)

# Кросс-валидация
cv_mae_scores = -cross_val_score(model_cb, X_train_scaled, y_train, cv=CV_FOLDS,
                                  scoring='neg_mean_absolute_error', n_jobs=-1)
cv_r2_scores = cross_val_score(model_cb, X_train_scaled, y_train, cv=CV_FOLDS,
                                scoring='r2', n_jobs=-1)

# Обучение на всей выборке
model_cb.fit(X_train_scaled, y_train, verbose=False)
y_train_pred_cb = model_cb.predict(X_train_scaled)

results.append({
    'model_name': 'CatBoostRegressor',
    'train_mae': mean_absolute_error(y_train, y_train_pred_cb),
    'train_r2': r2_score(y_train, y_train_pred_cb),
    'cv_mae_mean': cv_mae_scores.mean(),
    'cv_mae_std': cv_mae_scores.std(),
    'cv_r2_mean': cv_r2_scores.mean(),
    'cv_r2_std': cv_r2_scores.std(),
})

logger.info("Результаты:")
last_model = results[-1]
logger.info(f"Модель: {last_model['model_name']}")
for key, value in last_model.items():
    if key != 'model_name':
        logger.info(f"------- {key}: {value:.4f}" if isinstance(value, float) else f"------- {key}: {value}")

катбуст показал почти нужные результаты

### lightgbm

In [ ]:
lgb_params = {
    'n_estimators': 500,
    'max_depth': 6,
    'learning_rate': 0.1,
    'objective': 'regression_l1',
    'random_state': RANDOM_STATE,
    'verbosity': -1,
    'n_jobs': -1
}
model_lgb = lgb.LGBMRegressor(**lgb_params)

# Кросс-валидация
cv_mae_scores = -cross_val_score(model_lgb, X_train_scaled, y_train, cv=CV_FOLDS,
                                  scoring='neg_mean_absolute_error', n_jobs=-1)
cv_r2_scores = cross_val_score(model_lgb, X_train_scaled, y_train, cv=CV_FOLDS,
                                scoring='r2', n_jobs=-1)

# Обучение на всей выборке
model_lgb.fit(X_train_scaled, y_train)
y_train_pred_lgb = model_lgb.predict(X_train_scaled)

results.append({
    'model_name': 'LightGBMRegressor',
    'train_mae': mean_absolute_error(y_train, y_train_pred_lgb),
    'train_r2': r2_score(y_train, y_train_pred_lgb),
    'cv_mae_mean': cv_mae_scores.mean(),
    'cv_mae_std': cv_mae_scores.std(),
    'cv_r2_mean': cv_r2_scores.mean(),
    'cv_r2_std': cv_r2_scores.std(),
})

logger.info("Результаты:")
last_model = results[-1]
logger.info(f"Модель: {last_model['model_name']}")
for key, value in last_model.items():
    if key != 'model_name':
        logger.info(f"------- {key}: {value:.4f}" if isinstance(value, float) else f"------- {key}: {value}")

### нейронка

In [ ]:
def set_seed(seed=20):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(20)

In [ ]:
X_train_full, X_val, y_train_full, y_val = train_test_split(
    X_train_scaled, y_train, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

In [ ]:
class RegressionNN(nn.Module):
    def __init__(self, input_dim, hidden_dims=[256, 128, 64], activation='relu', dropout_rate=0.2, use_batchnorm=True):
        super().__init__()
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(hidden_dim))
            
            if activation == 'relu':
                layers.append(nn.ReLU())
            elif activation == 'elu':
                layers.append(nn.ELU())
            elif activation == 'selu':
                layers.append(nn.SELU())
            elif activation == 'tanh':
                layers.append(nn.Tanh())
            
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

In [ ]:
HIDDEN_DIMS = [256, 128, 64]
ACTIVATION = 'relu'
DROPOUT_RATE = 0.2
USE_BATCHNORM = True
LEARNING_RATE = 0.001
BATCH_SIZE = 32
WEIGHT_DECAY = 1e-5
EPOCHS = 500
EARLY_STOP = 50

In [ ]:
X_train_t = torch.tensor(X_train_full.values, dtype=torch.float32).to(DEVICE)
X_val_t = torch.tensor(X_val.values, dtype=torch.float32).to(DEVICE)

y_train_t = torch.tensor(y_train_full.values, dtype=torch.float32).to(DEVICE).view(-1, 1)
y_val_t = torch.tensor(y_val.values, dtype=torch.float32).to(DEVICE).view(-1, 1)

In [ ]:
model_nn = RegressionNN(
    input_dim=X_train_t.shape[1],
    hidden_dims=HIDDEN_DIMS,
    activation=ACTIVATION,
    dropout_rate=DROPOUT_RATE,
    use_batchnorm=USE_BATCHNORM
).to(DEVICE)

optimizer = optim.Adam(model_nn.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.L1Loss()  # MAE

train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset = TensorDataset(X_val_t, y_val_t)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_mae_scores = []
cv_r2_scores = []

# Оборачиваем итератор по фолдам в tqdm
for fold, (train_idx, val_idx) in enumerate(tqdm(kf.split(X_train_scaled), total=CV_FOLDS, desc='CV Folds')):
    X_tr = X_train_scaled.iloc[train_idx].values
    y_tr = y_train.iloc[train_idx].values
    X_val_fold = X_train_scaled.iloc[val_idx].values
    y_val_fold = y_train.iloc[val_idx].values

    X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(DEVICE)
    y_tr_t = torch.tensor(y_tr, dtype=torch.float32).to(DEVICE).view(-1, 1)
    X_val_t_fold = torch.tensor(X_val_fold, dtype=torch.float32).to(DEVICE)
    y_val_t_fold = torch.tensor(y_val_fold, dtype=torch.float32).to(DEVICE).view(-1, 1)

    model_fold = RegressionNN(
        input_dim=X_tr_t.shape[1],
        hidden_dims=HIDDEN_DIMS,
        activation=ACTIVATION,
        dropout_rate=DROPOUT_RATE,
        use_batchnorm=USE_BATCHNORM
    ).to(DEVICE)

    optimizer_fold = optim.Adam(model_fold.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion_fold = nn.L1Loss()

    train_dataset_fold = TensorDataset(X_tr_t, y_tr_t)
    val_dataset_fold = TensorDataset(X_val_t_fold, y_val_t_fold)

    train_loader_fold = DataLoader(train_dataset_fold, batch_size=BATCH_SIZE, shuffle=True)
    val_loader_fold = DataLoader(val_dataset_fold, batch_size=BATCH_SIZE, shuffle=False)

    best_val_loss = float('inf')
    patience_counter = 0
    best_state_fold = None

    for epoch in range(EPOCHS):
        model_fold.train()
        train_loss = 0
        for batch_X, batch_y in train_loader_fold:
            optimizer_fold.zero_grad()
            pred = model_fold(batch_X).squeeze()
            loss = criterion_fold(pred, batch_y)
            loss.backward()
            optimizer_fold.step()
            train_loss += loss.item() * len(batch_X)
        train_loss /= len(train_dataset_fold)

        model_fold.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_X, batch_y in val_loader_fold:
                pred = model_fold(batch_X).squeeze()
                loss = criterion_fold(pred, batch_y)
                val_loss += loss.item() * len(batch_X)
        val_loss /= len(val_dataset_fold)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state_fold = model_fold.state_dict()
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP:
                break

    if best_state_fold:
        model_fold.load_state_dict(best_state_fold)

    model_fold.eval()
    with torch.no_grad():
        y_val_pred = model_fold(X_val_t_fold).cpu().numpy().flatten()
    cv_mae_scores.append(best_val_loss)
    cv_r2_scores.append(r2_score(y_val_fold, y_val_pred))

cv_mae_mean = np.mean(cv_mae_scores)
cv_mae_std = np.std(cv_mae_scores)
cv_r2_mean = np.mean(cv_r2_scores)
cv_r2_std = np.std(cv_r2_scores)

In [ ]:
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

for epoch in range(EPOCHS):
    model_nn.train()
    train_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        pred = model_nn(batch_X).squeeze()
        loss = criterion(pred, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(batch_X)
    train_loss /= len(train_dataset)

    model_nn.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            pred = model_nn(batch_X).squeeze()
            loss = criterion(pred, batch_y)
            val_loss += loss.item() * len(batch_X)
    val_loss /= len(val_dataset)

    if (epoch + 1) % 100 == 0:
        logger.info(f"Эпоха {epoch+1}/{EPOCHS}, Train MAE: {train_loss:.4f}, Val MAE: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = model_nn.state_dict()
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP:
            logger.info(f"Ранняя остановка на эпохе {epoch+1}")
            break

if best_model_state:
    model_nn.load_state_dict(best_model_state)

In [ ]:
model_nn.eval()
with torch.no_grad():
    y_train_pred = model_nn(X_train_t).cpu().numpy().flatten()

train_mae_nn = mean_absolute_error(y_train_full, y_train_pred)
train_r2_nn = r2_score(y_train_full, y_train_pred)

results.append({
    'model_name': 'NeuralNetwork_PyTorch',
    'train_mae': train_mae_nn,
    'train_r2': train_r2_nn,
    'cv_mae_mean': cv_mae_mean,
    'cv_mae_std': cv_mae_std,
    'cv_r2_mean': cv_r2_mean,
    'cv_r2_std': cv_r2_std
})

logger.info("Результаты:")
last_model = results[-1]
logger.info(f"Модель: {last_model['model_name']}")
for key, value in last_model.items():
    if key != 'model_name':
        logger.info(f"------- {key}: {value:.4f}")

нейронка не дала нам нормальных результов, возможно надо дать больше эпох к обучению и раннюю остановку увеличить

### константная модель

In [ ]:
mean_y_train = y_train.mean()
y_train_pred_const = np.full_like(y_train, mean_y_train)
y_test_pred_const = np.full_like(y_test, mean_y_train)

results.append({
    'model_name': 'Constant_Mean',
    'train_mae': mean_absolute_error(y_train, y_train_pred_const),
    'train_r2': r2_score(y_train, y_train_pred_const),
})

logger.info("Результаты константной модели:")
last_model = results[-1]
logger.info(f"Модель: {last_model['model_name']}")
for key, value in last_model.items():
    if key != 'model_name':
        logger.info(f"------- {key}: {value:.4f}")

ну тут странно было ожидать каких-то вау результатов

## сравнение результатов

In [ ]:
# сравним результаты моделей
results_pd = pd.DataFrame(results)
display(results_pd)

Катбуст дал лучшие показатели  
Покрутим дальше именно катбуст

In [ ]:
def objective_catboost(trial):
    # Базовые параметры
    params = {
        'iterations': trial.suggest_int('iterations', 500, 3000, step=100),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-5, 1.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 0.0, 1.0),
        'grow_policy': trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise', 'Lossguide']),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),
        'boosting_type': trial.suggest_categorical('boosting_type', ['Plain', 'Ordered']),
        'allow_writing_files': False,
        'loss_function': 'MAE',
        'random_seed': RANDOM_STATE,
        'verbose': False,
        'thread_count': -1
    }
    
    # Для Lossguide добавляем max_leaves
    if params['grow_policy'] == 'Lossguide':
        params['max_leaves'] = trial.suggest_int('max_leaves', 16, 255)
    
    # Кросс-валидация
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)  # увеличил до 5
    cv_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_scaled)):
        X_tr = X_train_scaled.iloc[train_idx].values
        y_tr = y_train.iloc[train_idx].values
        X_val = X_train_scaled.iloc[val_idx].values
        y_val = y_train.iloc[val_idx].values
        
        model = CatBoostRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=(X_val, y_val),
            early_stopping_rounds=50,
            verbose=False
        )
        y_pred = model.predict(X_val)
        cv_scores.append(mean_absolute_error(y_val, y_pred))
        
        trial.report(cv_scores[-1], fold)
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return np.mean(cv_scores)

optuna.logging.set_verbosity(optuna.logging.WARNING)
study_cb = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=RANDOM_STATE),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=3, interval_steps=1),
    study_name='catboost_temperature_improved'
)

logger.info("Запуск оптимизации CatBoost с Optuna...")
study_cb.optimize(objective_catboost, n_trials=N_TRIALS_CB, show_progress_bar=True)

logger.info(f"Лучшее значение MAE на CV: {study_cb.best_value:.4f}")
logger.info("Лучшие параметры:")
for key, value in study_cb.best_params.items():
    logger.info(f"  {key}: {value}")

best_params_cb = study_cb.best_params.copy()
model_cb_final = CatBoostRegressor(**best_params_cb, loss_function='MAE', random_seed=RANDOM_STATE, verbose=False)
model_cb_final.fit(X_train_scaled, y_train, verbose=False)
y_train_pred_cb = model_cb_final.predict(X_train_scaled)
y_test_pred_cb = model_cb_final.predict(X_test_scaled)

train_mae_cb = mean_absolute_error(y_train, y_train_pred_cb)
test_mae_cb = mean_absolute_error(y_test, y_test_pred_cb)
train_r2_cb = r2_score(y_train, y_train_pred_cb)
test_r2_cb = r2_score(y_test, y_test_pred_cb)

results.append({
    'model_name': 'CatBoostRegressor_Optimized',
    'train_mae': train_mae_cb,
    'test_mae': test_mae_cb,
    'train_r2': train_r2_cb,
    'test_r2': test_r2_cb
})

logger.info("Результаты оптимизированной модели CatBoost:")
logger.info(f"------- train_mae: {train_mae_cb:.4f}")
logger.info(f"------- test_mae: {test_mae_cb:.4f}")
logger.info(f"------- train_r2: {train_r2_cb:.4f}")
logger.info(f"------- test_r2: {test_r2_cb:.4f}")

optuna.logging.set_verbosity(optuna.logging.INFO)